## Ontario Topic-Section Matching
### Goal
Rebuild `hansard_api_bill_segments_clean.json` using topic-section grouping instead of paragraph-level bill number matching.

The new rescraper (`fetch_ontario_hansard_via_api.py`) stores a `topic` field with every paragraph — the Hansard's own label for that section of debate, like `HEALTH INSURANCE AMENDMENT ACT`. 

The approach:
1. Group paragraphs by date and topic section
2. Compute Jaccard similarity between normalized topic and normalized bill title
3. If similarity exceeds threshold, keep ALL paragraphs in that topic section
4. This preserves sequential debate continuity and avoids cross-matching between similar bill titles

### Cell 1: Load data and verify topic field is present in new rescrape output

In [3]:
import json
import re
from collections import defaultdict

REPO = "/Users/caprice/MiscellaneousProjects/legislative-health-factuality"

with open(f"{REPO}/data/canada/ontario/hansard_api_bill_segments.json") as f:
    seg = json.load(f)

with open(f"{REPO}/data/canada/ontario/ontario_bill_index.json") as f:
    idx = json.load(f)

health = {k: v for k, v in idx.items() if v.get("health_tag")}
print(f"Health bills in index: {len(health)}")
print(f"Bills in segments file: {len(seg)}")

# Verify topic field is stored in new rescrape output
sample_bill = list(seg.keys())[0]
sample_date = list(seg[sample_bill].keys())[0]
sample_para = seg[sample_bill][sample_date][0]
print(f"\nSample paragraph type: {type(sample_para)}")
if isinstance(sample_para, dict):
    print(f"Fields available: {list(sample_para.keys())}")
    print(f"Topic field: {sample_para.get('topic', 'NO TOPIC')}")
    print(f"Sample text: {sample_para.get('text', '')[:100]}")
else:
    print("WARNING: Paragraphs are plain strings, topic field not stored. Need to re-run rescrape.")

Health bills in index: 329
Bills in segments file: 226

Sample paragraph type: <class 'dict'>
Fields available: ['text', 'speaker', 'topic', 'assignment_method']
Topic field: WORKERS' COMPENSATION AMENDMENT ACT
Sample text: Some of us are still here to protect the rights of the citizens of Ontario in spite of some of the t


### Cell 2: Normalization and Jaccard Similarity Functions

Normalize both the topic field and the bill title by:
- Converting to uppercase
- Removing punctuation
- Stripping stopwords that don't distinguish bills from each other
- Lemmatizing common suffixes (AMENDMENT → AMEND, SERVICES → SERVICE)

Jaccard similarity = |A ∩ B| / |A ∪ B| where A and B are sets of normalized words.
A high Jaccard score means the topic and bill title share most of the same meaningful words.

In [9]:
STOPWORDS = {
    "AN", "ACT", "TO", "THE", "OF", "AND", "FOR", "IN", "A", "BY",
    "OR", "AMEND", "PROVIDE", "RESPECTING", "CERTAIN", "MAKE",
    "REQUIRE", "ENSURE", "WITH", "RESPECT", "RELATED", "RELATING",
    "ENACT", "GOVERNING", "FURTHER", "VARIOUS", "AMENDING",
    "CONCERNING", "AUTHORIZE", "REGULATE", "THAT", "RENDERING",
    "UNDER", "MAY", "ARE", "REGULATING", "INSURED", "CHARGE",
    "ESTABLISH", "IMPOSING", "PERSONS", "PAYMENT", "RENDERING",
    "IMPOSE", "PROVIDING", "AUTHORIZING", "REQUIRING", "ENABLING"
}

LEMMA_MAP = {
    "AMENDMENT": "AMEND",
    "AMENDMENTS": "AMEND",
    "SERVICES": "SERVICE",
    "NURSES": "NURSE",
    "HOMES": "HOME",
    "WORKERS": "WORKER",
    "PROFESSIONS": "PROFESSION",
    "PROFESSIONALS": "PROFESSION",
    "PRACTITIONERS": "PRACTITIONER",
    "REGULATIONS": "REGULATION",
    "RIGHTS": "RIGHT",
    "BENEFITS": "BENEFIT",
    "PAYMENTS": "PAYMENT",
    "PERSONS": "PERSON",
    "AMOUNTS": "AMOUNT",
    "ACTS": "ACT",
}

def get_short_title(title):
    """Extract just the bill short title, truncating at first comma."""
    return title.split(",")[0].strip()

def normalize(text):
    if not text:
        return set()
    text = text.upper()
    text = re.sub(r"[^A-Z\s]", " ", text)
    words = text.split()
    words = [LEMMA_MAP.get(w, w) for w in words]
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return set(words)

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

# Test short titles
for bk in ["1985-Bill113", "1985-Bill94", "1985-Bill32"]:
    short = get_short_title(idx[bk]["title"])
    print(f"{bk}: {short}")
    print(f"  Norm: {normalize(short)}")
    print()

# Test on a few examples
test_cases = [
    ("WORKERS COMPENSATION AMENDMENT ACT", "An Act to amend the Workers' Compensation Act"),
    ("HEALTH INSURANCE AMENDMENT ACT", "An Act to amend the Health Insurance Act"),
    ("HEALTH PROTECTION ACT", "An Act to amend the Health Insurance Act"),  # should be low
    ("NURSING HOMES ACT", "An Act to amend the Nursing Homes Act"),
    ("BUDGET DEBATE", "An Act to amend the Health Protection Act"),  # should be very low
]

print("Jaccard similarity tests:")
print(f"{'Topic':<45} {'Bill title':<50} {'Score':>6}")
print("-" * 105)
for topic, title in test_cases:
    score = jaccard(normalize(topic), normalize(title))
    print(f"{topic:<45} {title[:48]:<50} {score:>6.3f}")

1985-Bill113: An Act to amend the Homemakers and Nurses Services Act
  Norm: {'HOMEMAKERS', 'NURSE', 'SERVICE'}

1985-Bill94: An Act regulating the Amounts that Persons may charge for rendering Services that are Insured Services under the Health Insurance Act
  Norm: {'HEALTH', 'PERSON', 'AMOUNT', 'INSURANCE', 'SERVICE'}

1985-Bill32: An Act to amend the Tobacco Tax Act
  Norm: {'TOBACCO', 'TAX'}

Jaccard similarity tests:
Topic                                         Bill title                                          Score
---------------------------------------------------------------------------------------------------------
WORKERS COMPENSATION AMENDMENT ACT            An Act to amend the Workers' Compensation Act       1.000
HEALTH INSURANCE AMENDMENT ACT                An Act to amend the Health Insurance Act            1.000
HEALTH PROTECTION ACT                         An Act to amend the Health Insurance Act            0.333
NURSING HOMES ACT                             An Ac

### Cell 3: Explore topic-to-bill matching across all segments

For every topic that appears in the segments file, compute Jaccard similarity
against all health bill titles and find the best match. This tells us:
- What threshold separates real matches from false positives
- Which bills are matched by topic vs by bill number
- Which bills still have no match after topic matching

In [10]:
# Get all unique topics across all segments
all_topics = set()
for bill_key, dates in seg.items():
    for date, paras in dates.items():
        for p in paras:
            topic = p.get("topic", "") if isinstance(p, dict) else ""
            if topic:
                all_topics.add(topic)

print(f"Unique topics in segments: {len(all_topics)}")

# For each health bill, find best matching topic
THRESHOLD = 0.6
results = []
for bill_key, bill_info in health.items():
    title = bill_info.get("title", "")
    title_norm = normalize(get_short_title(title))
    
    best_topic = None
    best_score = 0.0
    for topic in all_topics:
        score = jaccard(normalize(topic), title_norm)
        if score > best_score:
            best_score = score
            best_topic = topic
    
    results.append({
        "bill_key": bill_key,
        "title": title[:60],
        "best_topic": best_topic,
        "best_score": best_score,
        "matched": best_score >= THRESHOLD
    })

matched = [r for r in results if r["matched"]]
unmatched = [r for r in results if not r["matched"]]

print(f"\nThreshold: {THRESHOLD}")
print(f"Bills matched by topic: {len(matched)}/{len(health)}")
print(f"Bills unmatched: {len(unmatched)}")

print("\n=== Sample matched ===")
for r in sorted(matched, key=lambda x: x["best_score"])[:5]:
    print(f"  {r['bill_key']} | score={r['best_score']:.3f}")
    print(f"    Title: {r['title']}")
    print(f"    Topic: {r['best_topic']}")

print("\n=== Sample unmatched ===")
for r in unmatched[:10]:
    print(f"  {r['bill_key']} | best_score={r['best_score']:.3f} | best_topic: {r['best_topic']}")
    print(f"    Title: {r['title']}")

Unique topics in segments: 723

Threshold: 0.6
Bills matched by topic: 39/329
Bills unmatched: 290

=== Sample matched ===
  1993-Bill165 | score=0.600
    Title: An Act to amend the Workers' Compensation Act and the Occupa
    Topic: OCCUPATIONAL HEALTH AND SAFETY AMENDMENT ACT
  1993-Bill157 | score=0.600
    Title: An Act to amend the Occupational Health and Safety Act / Pro
    Topic: OCCUPATIONAL HEALTH AND SAFETY AMENDMENT ACT
  2009-Bill160 | score=0.600
    Title: An Act to amend the Occupational Health and Safety Act and t
    Topic: OCCUPATIONAL HEALTH AND SAFETY AMENDMENT ACT
  2021-Bill263 | score=0.600
    Title: An Act to amend the Health Protection and Promotion Act to p
    Topic: HEALTH PROTECTION AND PROMOTION AMENDMENT ACT
  1993-Bill76 | score=0.667
    Title: An Act to improve environmental protection, increase account
    Topic: ENVIRONMENTAL PROTECTION AMENDMENT ACT

=== Sample unmatched ===
  1985-Bill19 | best_score=0.095 | best_topic: SERVICES FOR PERSONS WITH

In [11]:
# How many of the 226 bills currently in segments use bill_number assignment vs topic?
method_counts = {"bill_number_in_text": 0, "topic_keyword_match": 0, "other": 0}
for bill_key, dates in seg.items():
    for date, paras in dates.items():
        for p in paras:
            if isinstance(p, dict):
                method = p.get("assignment_method", "other")
                method_counts[method] = method_counts.get(method, 0) + 1

print("Assignment methods in current segments:")
for method, count in sorted(method_counts.items(), key=lambda x: -x[1]):
    print(f"  {method}: {count:,}")

Assignment methods in current segments:
  bill_number_in_text: 82,531
  topic_keyword_match: 6,390
  other: 0


### Cell 4: Build clean corpus using topic-section grouping

For bills already in segments, group paragraphs by topic section within each date.
Keep all paragraphs in a section if:
1. The bill number appears anywhere in that section (strongest signal), OR
2. Jaccard similarity between normalized topic and normalized bill title >= 0.8

In [ ]:
import re as _re

BILL_RE = re.compile(r"\bBill\s+(\d+)\b", re.IGNORECASE)

def build_clean_corpus(seg, idx, health, threshold=0.8):
    clean = {}
    stats = {"bill_number": 0, "topic_match": 0, "dropped": 0}
    
    for bill_key, dates in seg.items():
        if bill_key not in health:
            continue
        bnum = str(idx[bill_key].get("bill_number", ""))
        title_norm = normalize(get_short_title(idx[bill_key].get("title", "")))
        
        for date, paras in dates.items():
            # Group by topic section
            topic_groups = defaultdict(list)
            for p in paras:
                topic = p.get("topic", "") if isinstance(p, dict) else ""
                topic_groups[topic].append(p)
            
            for topic, group_paras in topic_groups.items():
                all_text = " ".join(
                    p.get("text", "") if isinstance(p, dict) else p
                    for p in group_paras
                )
                
                # Check bill number in section
                bill_in_text = bnum and bnum in BILL_RE.findall(all_text)
                
                # Check topic Jaccard match
                topic_score = jaccard(normalize(topic), title_norm)
                topic_match = topic_score >= threshold
                
                if bill_in_text or topic_match:
                    method = "bill_number" if bill_in_text else "topic_jaccard"
                    stats[method if method in stats else "topic_match"] += len(group_paras)
                    if bill_key not in clean:
                        clean[bill_key] = {}
                    if date not in clean[bill_key]:
                        clean[bill_key][date] = []
                    clean[bill_key][date].extend(group_paras)
                else:
                    stats["dropped"] += len(group_paras)
    
    return clean, stats

clean, stats = build_clean_corpus(seg, idx, health, threshold=0.8)

total_paras = sum(len(p) for d in clean.values() for p in d.values())
print(f"Clean corpus: {len(clean)} bills, {total_paras:,} paragraphs")
print(f"Stats: {stats}")

# Compare to old clean corpus
print(f"\nPrevious clean corpus: 249 bills, 83,443 paragraphs (bill number only)")
print(f"New clean corpus: {len(clean)} bills, {total_paras:,} paragraphs (bill number + topic Jaccard)")

### Cell 5: Section Anchor Extraction

Find the paragraph that formally introduces each bill under Orders of the Day.
This is the section marker for Ontario — equivalent to Quebec's "Étude du projet de loi n° X" headers.

Patterns that signal a bill section opening:
- "Resuming debate on Bill X..."
- "Second reading of Bill X..."  
- "Consideration of Bill X..."
- "Mr. X moved second reading of Bill X..."
- "Resuming the adjourned debate on the motion for second reading of Bill X..."

Once the anchor is found, keep ALL subsequent paragraphs until:
- A new bill anchor appears (different bill number)
- A new typeOfBusiness topic appears
- End of the date's content

Then apply Jaccard similarity as secondary verification:
the topic field of the anchor paragraph must score >= 0.8 against the bill title.

In [19]:
import re

ANCHOR_RE = re.compile(
    r"(?:Resuming|second reading of|Consideration of|moved second reading of|"
    r"adjourned debate on the motion for second reading of|"
    r"debate on the motion for second reading of|"
    r"motion for second reading of)"
    r".{0,60}Bill\s+(\d+)",
    re.IGNORECASE
)

def extract_sections_by_anchor(date_paras, bill_number, bill_title_norm, threshold=0.8):
    """
    Extract bill-specific paragraphs using section anchor approach.
    
    1. Find paragraphs that formally introduce Bill X
    2. Keep all paragraphs from that anchor until the next bill anchor
    3. Verify section topic matches bill title via Jaccard >= threshold
    """
    results = []
    in_section = False
    current_section = []
    current_topic = None
    
    for p in date_paras:
        text = p.get("text", "") if isinstance(p, dict) else p
        topic = p.get("topic", "") if isinstance(p, dict) else ""
        
        # Check if this paragraph is an anchor for ANY bill
        anchor_match = ANCHOR_RE.search(text)
        
        if anchor_match:
            found_bill = anchor_match.group(1)
            
            # If we were in a section, save it
            if in_section and current_section:
                # Verify topic matches via Jaccard
                topic_score = jaccard(normalize(current_topic or ""), bill_title_norm)
                if topic_score >= threshold or f"Bill {bill_number}" in " ".join(
                    p.get("text","") if isinstance(p,dict) else p 
                    for p in current_section
                ):
                    results.extend(current_section)
                current_section = []
            
            # Start new section if this is our bill
            if found_bill == bill_number:
                in_section = True
                current_topic = topic
                current_section = [p]
            else:
                in_section = False
        
        elif in_section:
            # Continue current section — but stop if topic changes to something unrelated
            if topic and current_topic and topic != current_topic:
                topic_score = jaccard(normalize(topic), bill_title_norm)
                if topic_score < 0.3:
                    # Topic changed to something unrelated — end section
                    if current_section:
                        verify_score = jaccard(normalize(current_topic or ""), bill_title_norm)
                        if verify_score >= threshold or f"Bill {bill_number}" in " ".join(
                            p.get("text","") if isinstance(p,dict) else p
                            for p in current_section
                        ):
                            results.extend(current_section)
                    current_section = []
                    in_section = False
                    current_topic = topic
                else:
                    current_section.append(p)
            else:
                current_section.append(p)
    
    # Save final section
    if in_section and current_section:
        verify_score = jaccard(normalize(current_topic or ""), bill_title_norm)
        if verify_score >= threshold or f"Bill {bill_number}" in " ".join(
            p.get("text","") if isinstance(p,dict) else p
            for p in current_section
        ):
            results.extend(current_section)
    
    return results


# Test on Bill 94 (extra billing) — known good bill
bk = "1985-Bill94"
bnum = str(idx[bk]["bill_number"])
title_norm = normalize(get_short_title(idx[bk]["title"]))
date = "1986-01-14"
paras = seg[bk][date]

print(f"Testing: {bk} on {date}")
print(f"Total paragraphs on date: {len(paras)}")
extracted = extract_sections_by_anchor(paras, bnum, title_norm)
print(f"Extracted via anchor: {len(extracted)} paragraphs")
print()
print("First 3 extracted paragraphs:")
for p in extracted[:3]:
    text = p.get("text","") if isinstance(p,dict) else p
    print(f"  {text[:150]}")
    print()

Testing: 1985-Bill94 on 1986-01-14
Total paragraphs on date: 52
Extracted via anchor: 23 paragraphs

First 3 extracted paragraphs:
  Hon. Mr. Elston moved second reading of Bill 94, An Act regulating the Amounts that Persons may charge for rendering Services that are Insured Service

  In fairness to the member for York Mills, who just vacated the House, I presume she knows a great deal about the issue and is probably going out now t

  Was that a different one? I miss the member for Brampton (Mr. Callahan). Perhaps he will be here in the evening session.



# For each session in PARLIAMENT_SESSIONS:
#   Query API by date range (start to end)
#   Get all Orders of the Day paragraphs in batches of 500
#   Group by topic
#   Find anchor paragraphs for each bill
#   Keep all paragraphs in anchor's topic section
#   Verify with Jaccard

In [32]:
zero_sessions = {
    "39-2":("2009-03-02","2011-10-06"),
    "40-1":("2011-11-30","2012-12-20"),
    "40-2":("2013-02-19","2014-05-02"),
    "41-1":("2014-07-02","2015-12-10"),
    "41-2":("2016-02-22","2017-12-07"),
    "41-3":("2018-03-19","2018-05-09"),
    "42-2":("2020-02-18","2022-05-04"),
    "44-1":("2025-04-14","2026-09-21"),
}

for session,(start,end) in zero_sessions.items():
    tob = "Orders of the Day" if session in {"42-1","42-2","43-1","44-1"} else "ORDERS OF THE DAY"
    try:
        resp = requests.post(API, headers=H, json={
            "keyword":"*",
            "filter":f"date ge {start}T00:00:00Z and date le {end}T23:59:59Z and typeOfBusiness eq '{tob}'",
            "rows":1
        }, timeout=15)
        count = resp.json().get("@odata.count",0)
        print(f"{session}: {count:,}")
        time.sleep(1)
    except Exception as e:
        print(f"{session}: error")

39-2: 22,641
40-1: 9,126
40-2: 15,776
41-1: 2,704
41-2: 56
41-3: 34
42-2: 28,198
44-1: 14,477
